# 04 — Sales Revenue Forecast Analysis

Visualize the Prophet time-series model — actual vs predicted, seasonality components, and 6-month forecast.

## Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, os, sys
from sqlalchemy import create_engine, text
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL, MODELS_DIR
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (13, 5)
print("✅ Setup complete")


## 1. Load Model & Data

In [ ]:

model = joblib.load(os.path.join(MODELS_DIR, 'forecast_model.pkl'))

sql = '''
SELECT DATE_TRUNC('month', order_date::date) AS ds,
       ROUND(SUM(revenue)::numeric,2)        AS y
FROM olist.fact_orders
GROUP BY DATE_TRUNC('month', order_date::date)
ORDER BY ds
'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)

df['ds'] = pd.to_datetime(df['ds'])
print(f"Historical data: {len(df)} months")
print(f"Date range: {df['ds'].min().date()} → {df['ds'].max().date()}")
print(f"Avg monthly revenue: ${df['y'].mean():,.2f}")


## 2. Forecast Next 6 Months

In [ ]:

future   = model.make_future_dataframe(periods=6, freq='MS')
forecast = model.predict(future)

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(forecast['ds'],
                forecast['yhat_lower'], forecast['yhat_upper'],
                alpha=0.2, color='steelblue', label='95% Confidence Interval')
ax.plot(forecast['ds'], forecast['yhat'], color='steelblue', lw=2, label='Forecast')
ax.scatter(df['ds'], df['y'], color='black', s=20, zorder=5, label='Actual')
cutoff = df['ds'].max()
ax.axvline(cutoff, color='red', linestyle='--', lw=1, label='Forecast Start')
ax.set_title('Sales Revenue Forecast — Prophet Model')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.legend()
plt.tight_layout()
plt.show()

future_only = forecast[forecast['ds'] > cutoff][['ds','yhat','yhat_lower','yhat_upper']]
print("
6-Month Revenue Forecast:")
for _, r in future_only.iterrows():
    print(f"  {r['ds'].strftime('%Y-%m')}  →  ${r['yhat']:>12,.2f}  [${r['yhat_lower']:,.2f} – ${r['yhat_upper']:,.2f}]")


## 3. Seasonality Components

In [ ]:

fig = model.plot_components(forecast)
fig.suptitle('Prophet Model Components — Trend & Seasonality', y=1.02)
plt.tight_layout()
plt.show()


## 4. Actual vs Predicted

In [ ]:

hist_forecast = forecast[forecast['ds'].isin(df['ds'])].merge(df, on='ds')
hist_forecast['error_pct'] = abs(hist_forecast['yhat'] - hist_forecast['y']) / hist_forecast['y'] * 100
mape = hist_forecast['error_pct'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_forecast['ds'], hist_forecast['y']/1e6,     label='Actual',    color='black', lw=2)
axes[0].plot(hist_forecast['ds'], hist_forecast['yhat']/1e6,  label='Predicted', color='steelblue', lw=2, linestyle='--')
axes[0].set_title(f'Actual vs Predicted Revenue (MAPE: {mape:.1f}%)')
axes[0].set_ylabel('Revenue ($M)')
axes[0].legend()
axes[1].bar(hist_forecast['ds'], hist_forecast['error_pct'], color='steelblue', alpha=0.7)
axes[1].axhline(mape, color='red', linestyle='--', label=f'Avg MAPE: {mape:.1f}%')
axes[1].set_title('Prediction Error % by Month')
axes[1].set_ylabel('Error %')
axes[1].legend()
plt.tight_layout()
plt.show()
